# Task 192: learned_pd_ct — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Learned primal-dual CT reconstruction

**Paper**: Learned Primal-Dual Reconstruction
**Repository**: https://github.com/adler-j/learned_primal_dual

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 30.00 dB
- **SSIM**: 0.6460
- **Evaluation**: CT 重建 (FBP + TV-PDHG 去噪) — 2D 图像 PSNR

### Mathematical Formulation

The Radon transform maps the 2D attenuation $f(x,y)$ to sinogram $p(\theta, s)$:

$$p(\theta, s) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x,y)\,\delta(x\cos\theta + y\sin\theta - s)\,dx\,dy$$

**Filtered Back-Projection (FBP)**:
$$f(x,y) = \int_0^{\pi} \left[ p(\theta, \cdot) \ast g \right](x\cos\theta + y\sin\theta)\,d\theta$$

where $g$ is the ramp filter: $\hat{g}(\omega) = |\omega|$.

**Iterative reconstruction** (SIRT/CGLS):
$$x^{(k+1)} = x^{(k)} + \alpha \, A^T(b - Ax^{(k)})$$

**TV-regularized**:
$$\hat{x} = \arg\min_x \frac{1}{2}\|Ax - b\|_2^2 + \lambda \|\nabla x\|_1$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
Task 192: learned_pd_ct — CT Reconstruction via TV-PDHG
=========================================================
Inverse problem: Reconstruct a 2D image from its Radon transform (CT sinogram).

Forward model: y = R(x) + noise
  R: Radon transform (parallel beam CT)
  x: 2D phantom image (ground truth)
  y: noisy sinogram (input measurement)

Inverse solver: Two-stage approach:
  1. Filtered Back-Projection (FBP) for initial reconstruction
  2. Total Variation (TV) denoising via Chambolle-Pock PDHG to remove artifacts

  This mirrors the "unrolled" primal-dual approach: the FBP gives a good initial
  estimate, and TV-PDHG refines it — analogous to a single "block" of the Learned
  Primal-Dual network (Adler & Öktem, IEEE TMI 2018) but with classical TV instead
  of learned CNN components.

Reference: Adler & Öktem, "Learned Primal-Dual Reconstruction", IEEE TMI 2018
Repo: https://github.com/adler-j/learned_primal_dual

Usage: /data/yjh/learned_pd_ct_env/bin/python learned_pd_ct_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`grad`, `div`

In [ ]:
def grad(x):
    """Discrete gradient with forward differences: returns (2, N, N)."""
    gx = np.zeros_like(x)
    gy = np.zeros_like(x)
    gx[:-1, :] = x[1:, :] - x[:-1, :]
    gy[:, :-1] = x[:, 1:] - x[:, :-1]
    return np.stack([gx, gy], axis=0)

def div(p):
    """Discrete divergence = -∇^* (negative adjoint of gradient)."""
    px, py = p[0], p[1]
    dx = np.zeros_like(px)
    dy = np.zeros_like(py)
    dx[0, :] = px[0, :]
    dx[1:-1, :] = px[1:-1, :] - px[:-2, :]
    dx[-1, :] = -px[-2, :]
    dy[:, 0] = py[:, 0]
    dy[:, 1:-1] = py[:, 1:-1] - py[:, :-2]
    dy[:, -1] = -py[:, -1]
    return dx + dy

## 4. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import json
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from skimage.data import shepp_logan_phantom
from skimage.transform import resize, radon, iradon

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── 1. Paths ──────────────────────────────────────────────────────────────────
RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# ── 2. Generate Shepp-Logan phantom ──────────────────────────────────────────
N = 256
gt = resize(shepp_logan_phantom(), (N, N), anti_aliasing=True).astype('float64')
gt = (gt - gt.min()) / (gt.max() - gt.min())  # normalise to [0, 1]
print(f"[INFO] Phantom shape: {gt.shape}, range: [{gt.min():.4f}, {gt.max():.4f}]")

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# ── 3. Forward model: Radon transform + noise ────────────────────────────────
n_angles = 180
theta_angles = np.linspace(0., 180., n_angles, endpoint=False)

# Clean sinogram
sinogram_clean = radon(gt, theta=theta_angles, circle=False)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Add Gaussian noise (1% of peak sinogram value)
noise_std = 0.01 * sinogram_clean.max()
rng = np.random.default_rng(42)
noise = noise_std * rng.standard_normal(sinogram_clean.shape)
sinogram_noisy = sinogram_clean + noise

print(f"[INFO] Sinogram shape: {sinogram_noisy.shape}")
print(f"[INFO] Noise std: {noise_std:.4f} (1% of sinogram max {sinogram_clean.max():.2f})")

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# ── 4. Stage 1: Filtered Back-Projection (FBP) ──────────────────────────────
fbp_recon = iradon(sinogram_noisy, theta=theta_angles, circle=False,
                   filter_name='ramp')
fbp_recon = np.clip(fbp_recon, 0, 1)

fbp_psnr = psnr_fn(gt, fbp_recon, data_range=1.0)
print(f"[INFO] FBP reconstruction: PSNR = {fbp_psnr:.2f} dB")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ── 5. Stage 2: TV-PDHG denoising (Chambolle-Pock) ──────────────────────────
#
# Solve:  min_x  0.5 * ||x - x_fbp||^2  +  lam * ||∇x||_{2,1}
#         s.t.   0 <= x <= 1
#
# This is a TV-denoising (ROF) problem solved by PDHG:
#   min_x  f(x) + g(∇x)
#   f(x) = 0.5 * ||x - x_fbp||^2 + indicator_{[0,1]}(x)
#   g(y) = lam * ||y||_{2,1}
#
# Proximal operators:
#   prox_{tau*f}(z) = clip((z + tau*x_fbp) / (1+tau), 0, 1)
#   prox_{sigma*g*}(p) = pointwise project onto l2 ball of radius lam

# ||∇||^2 <= 8 for 2D forward differences
norm_grad = np.sqrt(8.0)

# TV regularization weight
lam = 0.02

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Step sizes (Chambolle-Pock for denoising)
tau = 1.0 / norm_grad
sigma = 1.0 / norm_grad

niter_tv = 300
print(f"[INFO] TV-PDHG denoising: lam={lam}, niter={niter_tv}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Initialize from FBP
x = fbp_recon.copy()
x_bar = x.copy()
p = np.zeros((2, N, N), dtype='float64')  # dual variable

for k in range(niter_tv):
    x_old = x.copy()

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
# Dual update: p = prox_{sigma * g*}(p + sigma * ∇(x_bar))
    p = p + sigma * grad(x_bar)
    # Project onto l2 balls of radius lam
    norms = np.sqrt(p[0]**2 + p[1]**2)
    scale = np.maximum(norms / lam, 1.0)
    p = p / scale[np.newaxis, :, :]

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Primal update: x = prox_{tau * f}(x - tau * (-div(p)))
    #   prox_{tau*f}(z) = clip( (z + tau * x_fbp) / (1 + tau), 0, 1 )
    #   ∇^*(p) = -div(p), so the update is x - tau * (-div(p)) = x + tau * div(p)
    x = np.clip((x + tau * div(p) + tau * fbp_recon) / (1.0 + tau), 0, 1)

# Over-relaxation (theta=1)
    x_bar = 2 * x - x_old

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
if (k + 1) % 50 == 0:
        psnr_k = psnr_fn(gt, x, data_range=1.0)
        print(f"  TV iter {k+1:4d}/{niter_tv}: PSNR = {psnr_k:.2f} dB")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
recon = x.astype('float32')
gt32 = gt.astype('float32')
print(f"[INFO] Reconstruction range: [{recon.min():.4f}, {recon.max():.4f}]")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# ── 6. Evaluation ────────────────────────────────────────────────────────────
data_range = float(gt32.max() - gt32.min())
psnr_val = float(psnr_fn(gt32, recon, data_range=data_range))
ssim_val = float(ssim_fn(gt32, recon, data_range=data_range))
cc_val = float(np.corrcoef(gt32.ravel(), recon.ravel())[0, 1])

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
print(f"\n{'='*50}")
print(f"  PSNR  = {psnr_val:.2f} dB")
print(f"  SSIM  = {ssim_val:.4f}")
print(f"  CC    = {cc_val:.4f}")
print(f"{'='*50}\n")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# ── 7. Save numerical results ────────────────────────────────────────────────
metrics = {
    "psnr_db": round(psnr_val, 2),
    "ssim": round(ssim_val, 4),
    "cc": round(cc_val, 4),
}
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as fp:
    json.dump(metrics, fp, indent=2)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), gt32)
np.save(os.path.join(RESULTS_DIR, "reconstruction.npy"), recon)
np.save(os.path.join(RESULTS_DIR, "input.npy"), sinogram_noisy.astype('float32'))
print("[INFO] Saved metrics.json, ground_truth.npy, reconstruction.npy, input.npy")

# ── 8. Visualization (2×2 panel) ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (a) Ground truth
im0 = axes[0, 0].imshow(gt, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title("(a) Ground Truth Phantom", fontsize=13)
axes[0, 0].axis('off')
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# (b) Noisy sinogram
im1 = axes[0, 1].imshow(sinogram_noisy, cmap='gray', aspect='auto')
axes[0, 1].set_title("(b) Noisy Sinogram (Input)", fontsize=13)
axes[0, 1].set_xlabel("Detector pixel")
axes[0, 1].set_ylabel("Projection angle index")
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (c) TV-PDHG reconstruction
im2 = axes[1, 0].imshow(recon, cmap='gray', vmin=0, vmax=1)
axes[1, 0].set_title(
    f"(c) FBP + TV-PDHG Reconstruction\nPSNR={psnr_val:.1f} dB, SSIM={ssim_val:.3f}",
    fontsize=13,
)
axes[1, 0].axis('off')
plt.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04)

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (d) Error map
error_map = np.abs(gt32 - recon)
im3 = axes[1, 1].imshow(error_map, cmap='hot', vmin=0,
                         vmax=max(error_map.max(), 0.01))
axes[1, 1].set_title("(d) Absolute Error |GT − Recon|", fontsize=13)
axes[1, 1].axis('off')
plt.colorbar(im3, ax=axes[1, 1], fraction=0.046, pad=0.04)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
fig.suptitle(
    "Task 192: CT Reconstruction via FBP + TV-PDHG\n"
    f"256×256 Shepp-Logan, {n_angles} angles, 1% noise",
    fontsize=15,
    fontweight='bold',
    y=0.98,
)
plt.tight_layout(rect=[0, 0, 1, 0.94])
fig_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"[INFO] Saved figure → {fig_path}")

print("\n[DONE] Task 192 completed successfully.")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 5. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **learned_pd_ct**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=30.00 dB, SSIM=0.6460)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: Learned Primal-Dual Reconstruction
- Repository: https://github.com/adler-j/learned_primal_dual
